# Bank Marketing Dataset — Essential Cleaning
Minimal cleaning only, prepared for LightGBM (which handles missing/categorical data natively).

Steps:
1. Load data
2. Inspect shape, dtypes, duplicates, nulls, unique values
3. Drop exact duplicate rows
4. Encode target `y` as 0/1
5. Cast categorical columns to `category` dtype
6. Save cleaned CSV

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)

## 1. Load data

In [ ]:
df = pd.read_csv('bank-direct-marketing-campaigns.csv')
df.shape

## 2. Inspect the data

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
# Duplicate rows
df.duplicated().sum()

In [ ]:
# True missing values (NaN) — note: 'unknown' is a placeholder category, not NaN
df.isnull().sum()

In [ ]:
# Unique values per categorical column (check for messy/inconsistent entries)
cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan',
            'contact', 'month', 'day_of_week', 'poutcome']

for c in cat_cols:
    print(c, '->', df[c].unique())

In [ ]:
# Target class balance
df['y'].value_counts()

## 3. Drop exact duplicate rows

In [ ]:
before = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
after = df.shape[0]
print(f'Dropped {before - after} duplicate rows. New shape: {df.shape}')

## 4. Encode target `y` as 0/1

In [ ]:
df['y'] = df['y'].map({'no': 0, 'yes': 1})
df['y'].value_counts()

## 5. Cast categorical columns to `category` dtype
This lets LightGBM use native categorical handling instead of guessing types or requiring one-hot encoding.

Note: `'unknown'` values are kept as-is (as their own category) — no imputation, since LightGBM handles this fine and imputing would be unnecessary preprocessing.

In [ ]:
for c in cat_cols:
    df[c] = df[c].astype('category')

df.dtypes

## 6. Save cleaned data

In [ ]:
df.to_csv('bank_marketing_cleaned.csv', index=False)
print('Saved: bank_marketing_cleaned.csv', df.shape)

## Notes for LightGBM training
- CSV does **not** preserve `category` dtype — re-cast `cat_cols` to `'category'` after `pd.read_csv()` when loading `bank_marketing_cleaned.csv` for training.
- Pass `categorical_feature=cat_cols` (or `'auto'`) to LightGBM's `Dataset`/`fit()`.
- Target is imbalanced (~88% no / 12% yes) — consider `is_unbalance=True` or `scale_pos_weight`.
- `pdays == 999` is a sentinel meaning 'not previously contacted' — left as-is intentionally, LightGBM will split on it naturally.